# Angular 16 — Complete Instructor Reference Guide

> **Release Date:** May 3, 2023 | **Codename:** N/A | **Type:** Major release

---

## At a Glance

Angular 16 is one of the most significant Angular releases in years, introducing the **Signals** reactive primitive, a preview of **server-side rendering hydration**, and a suite of developer experience improvements. It sets the foundation for Angular's future reactivity model.

---

## Version Requirements

| Dependency | Required Version |
|---|---|
| Node.js | 16.14+ or 18.10+ |
| TypeScript | 4.9.x – 5.0.x |
| RxJS | 6.5.3+ or 7.4+ |
| Zone.js | 0.13.x |
| Angular CLI | 16.x |

---

## Upgrade Command

```bash
# Update Angular CLI globally
npm install -g @angular/cli@16

# Update workspace (from Angular 15)
ng update @angular/core@16 @angular/cli@16

# Update Angular Material (if used)
ng update @angular/material@16
```

---

## Top 8 Features Summary

| # | Feature | Status | Impact |
|---|---|---|---|
| 1 | **Angular Signals** | Developer Preview | Revolutionary — new reactive primitive |
| 2 | **Required Inputs** | Stable | Compile-time input validation |
| 3 | **Router Input Binding** | Stable | No more `ActivatedRoute` injection |
| 4 | **SSR Non-Destructive Hydration** | Developer Preview | 45% faster SSR re-render |
| 5 | **`takeUntilDestroyed`** | Stable | Simplified subscription cleanup |
| 6 | **Vite + esbuild** | Developer Preview | 67% faster `ng serve` cold starts |
| 7 | **`@Input` with Transforms** | Stable | Convert string → number/boolean automatically |
| 8 | **Self-Closing Component Tags** | Stable | `<my-comp />` syntax in templates |

# Section 1 — Detailed Notes

---

## 1.1 Angular Signals (Developer Preview)

**What it is:** A new **reactive primitive** built into Angular that tracks state and automatically notifies consumers when state changes. Think of it as Angular's answer to Solid.js signals, Vue's reactivity, or MobX observables — but first-class in the framework.

**Core primitives:**

| Primitive | Description |
|---|---|
| `signal(value)` | Creates a readable/writable reactive state holder |
| `computed(() => ...)` | Creates a derived value that recalculates when deps change |
| `effect(() => ...)` | Side-effect that runs when any signal it reads changes |

**Why it matters:**
- Angular has relied on **Zone.js** for change detection for 8+ years. Zone.js monkey-patches browser APIs (setTimeout, Promises, etc.) to detect changes — it's powerful but adds overhead.
- Signals provide a **fine-grained reactivity model**: only components whose signals changed re-render, not the whole component tree.
- Signals are the foundation for **Zone-less Angular** (coming in future releases).

**Key signal methods:**
```
signal.set(value)        — replace value
signal.update(fn)        — update based on current value
signal.mutate(fn)        — mutate in place (for objects/arrays)
signal()                 — read the current value
```

**Signal vs Observable:**
| | Signal | Observable (RxJS) |
|---|---|---|
| Synchronous by default | ✅ Yes | ❌ No (async) |
| Auto-tracked in templates | ✅ Yes | ❌ No (needs async pipe) |
| Always has a value | ✅ Yes | ❌ No |
| Can be lazy | ❌ No | ✅ Yes |
| Composable with operators | ❌ Limited | ✅ Full RxJS power |

**Interop helpers (v16):**
- `toSignal(observable)` — converts an Observable to a Signal
- `toObservable(signal)` — converts a Signal to an Observable

---

## 1.2 Required Inputs

**What it is:** Angular 16 adds the `required: true` option to `@Input()`, causing a **compile-time error** if a parent component forgets to pass a required input.

**Before (Angular 15):**
- No way to enforce a required `@Input` at compile time
- Runtime errors: `Cannot read property 'name' of undefined`
- Had to add manual null checks everywhere

**After (Angular 16):**
```typescript
@Input({ required: true }) user!: User;
// TS error in parent template if [user] is missing
```

**Rules:**
- `required: true` triggers a **TypeScript template type-check error** — caught at build time
- Works with both class-based components and standalone components
- Cannot be combined with `@Input('alias')` syntax; use `@Input({ alias: '...', required: true })`
- `ngOnChanges` `SimpleChanges` correctly types required inputs as always-defined

---

## 1.3 Router Input Binding (`withComponentInputBinding`)

**What it is:** When `withComponentInputBinding()` is added to `provideRouter()`, the router **automatically binds route parameters, query parameters, and resolver data** to matching `@Input()` properties on the routed component.

**Sources that bind automatically:**
| Source | Example | Component Input |
|---|---|---|
| Path param | `/:id` in route | `@Input() id!: string` |
| Query param | `?tab=settings` | `@Input() tab?: string` |
| Resolve data | `resolve: { user: userResolver }` | `@Input() user!: User` |
| Static data | `data: { title: 'Home' }` | `@Input() title!: string` |

**Type coercion:** Route params are always strings. Use `@Input({ transform: numberAttribute })` to auto-convert to numbers.

---

## 1.4 SSR Non-Destructive Hydration (Developer Preview)

**What it is:** When Angular Universal renders HTML on the server, the browser previously **destroyed the server-rendered DOM** and rebuilt it from scratch on the client — causing flickering and wasted work.

**Non-destructive hydration:** Angular 16's hydration reuses the server-rendered DOM nodes and attaches Angular's event listeners and data bindings **without re-rendering**. The page is immediately interactive.

**Performance impact:**
- LCP (Largest Contentful Paint): up to 45% improvement
- No flickering / content layout shift on hydration
- Reduces amount of work the browser's main thread must do

**Limitations (v16 Developer Preview):**
- Does not support `ViewEncapsulation.ShadowDom`
- Does not work with `i18n` (yet)
- Requires `provideClientHydration()` in `bootstrapApplication`

---

## 1.5 `takeUntilDestroyed` and `DestroyRef`

**What it is:** Two new utilities for automatic subscription cleanup, eliminating the `Subject + takeUntil + ngOnDestroy` boilerplate pattern.

**`DestroyRef`** — injectable token that provides a callback when a component/directive/pipe/service is destroyed:
```typescript
const destroyRef = inject(DestroyRef);
destroyRef.onDestroy(() => { /* cleanup */ });
```

**`takeUntilDestroyed(destroyRef?)`** — RxJS operator that automatically completes an Observable when the current component/directive is destroyed:
```typescript
// BEFORE — Angular 15 pattern
private destroy$ = new Subject<void>();
ngOnInit()    { this.data$.pipe(takeUntil(this.destroy$)).subscribe(...); }
ngOnDestroy() { this.destroy$.next(); this.destroy$.complete(); }

// AFTER — Angular 16
ngOnInit() {
  this.data$.pipe(takeUntilDestroyed(this.destroyRef)).subscribe(...);
}
// No ngOnDestroy needed!
```

When called inside an injection context (constructor, field initializer), `destroyRef` parameter is optional:
```typescript
// Even cleaner — injection context auto-detects the component
readonly data = this.dataService.get().pipe(takeUntilDestroyed());
```

---

## 1.6 Vite + esbuild (Developer Preview)

**What it is:** Angular 16 introduces an **alternative build system** using Vite (dev server) and esbuild (bundler/compiler) as a Developer Preview, replacing the Webpack-based build system for `ng serve`.

**Performance comparison:**
| Operation | Webpack (v15) | Vite+esbuild (v16) | Improvement |
|---|---|---|---|
| Cold start (`ng serve`) | ~8–15s | ~1–3s | ~67% faster |
| Hot Module Replacement | ~2–5s | ~300ms | ~85% faster |
| Production build | Baseline | ~25–40% faster | Significant |

**Enable in `angular.json`:**
```json
{
  "architect": {
    "build": {
      "builder": "@angular-devkit/build-angular:application"
    },
    "serve": {
      "builder": "@angular-devkit/build-angular:dev-server"
    }
  }
}
```

**Note:** The new `application` builder (replacing `browser`) becomes stable in Angular 17.

---

## 1.7 `@Input` with Transforms

**What it is:** Route parameters (and HTML attributes) are always strings. Angular 16 adds a `transform` option to `@Input()` that automatically converts the string value to the correct type.

**Built-in transform functions:**
| Function | Converts | Example |
|---|---|---|
| `numberAttribute` | `"42"` → `42` | `@Input({ transform: numberAttribute }) count!: number` |
| `booleanAttribute` | `""` / `"true"` → `true`, absent → `false` | `@Input({ transform: booleanAttribute }) disabled!: boolean` |

**Custom transform:**
```typescript
@Input({ transform: (v: string) => v.split(',') }) tags!: string[];
```

**Works with Router Input Binding:**
```html
<!-- Route: /products?page=2&count=20 -->
<app-products page="2" count="20"> <!-- both route-bound as strings -->
```
```typescript
@Input({ transform: numberAttribute }) page = 1;
@Input({ transform: numberAttribute }) count = 10;
// Automatically: page = 2, count = 20 (numbers, not strings)
```

---

## 1.8 Self-Closing Component Tags

**What it is:** Angular 16 adds support for **self-closing syntax** for components with no projected content. Previously, all component tags required an explicit closing tag.

```html
<!-- Before Angular 16 -->
<app-spinner></app-spinner>
<app-icon name="home"></app-icon>
<my-badge count="5"></my-badge>

<!-- After Angular 16 — self-closing is valid -->
<app-spinner />
<app-icon name="home" />
<my-badge count="5" />
```

**Rules:**
- Only valid when the component has **no `<ng-content>`** to project into, or when nothing is being projected
- Standard HTML void elements (`<input>`, `<br>`, etc.) already worked; this extends to Angular components
- Works in all template locations: component templates, structural directives, etc.

---

## 1.9 Jest Support (Developer Preview)

**What it is:** Angular 16 adds experimental Jest support as an alternative to Karma + Jasmine. Jest is the industry-standard test runner for JavaScript projects.

**Enable in `angular.json`:**
```json
{
  "test": {
    "builder": "@angular-devkit/build-angular:jest",
    "options": {
      "tsConfig": "tsconfig.spec.json"
    }
  }
}
```

**Benefits over Karma:**
- No browser required (runs in Node.js with jsdom)
- Faster test execution
- Better snapshot testing
- Industry-standard — familiar to React/Vue developers
- Better CI/CD integration (no Headless Chrome needed)

---

## 1.10 TypeScript 5.0 Support

**What it is:** Angular 16 adds support for **TypeScript 5.0**, which brings:
- Decorators (Stage 3 — official TC39 spec, not TypeScript-proprietary anymore)
- `const` type parameters
- `resolution-mode` for `moduleResolution`
- Performance improvements (~10-15% faster type checking)

**Important:** Angular 16 also still supports TypeScript 4.9.x for compatibility.

# Section 2 — Code Examples

---

## Example 1: Angular Signals — Counter and Derived State

```typescript
// counter.component.ts
import { Component, signal, computed, effect } from '@angular/core';

@Component({
  selector: 'app-counter',
  standalone: true,
  template: `
    <div class="counter">
      <h2>Count: {{ count() }}</h2>
      <h3>Double: {{ doubleCount() }}</h3>
      <h3>Status: {{ status() }}</h3>

      <button (click)="increment()">+1</button>
      <button (click)="decrement()">-1</button>
      <button (click)="reset()">Reset</button>
    </div>
  `
})
export class CounterComponent {
  // 1. Writable signal — mutable state
  count = signal(0);

  // 2. Computed signal — auto-recalculates when 'count' changes
  doubleCount = computed(() => this.count() * 2);

  status = computed(() => {
    const n = this.count();
    if (n < 0)  return 'Negative';
    if (n === 0) return 'Zero';
    return n > 10 ? 'High' : 'Positive';
  });

  constructor() {
    // 3. Effect — runs whenever 'count' changes
    effect(() => {
      console.log(`Count changed to: ${this.count()}`);
      // No need to manually unsubscribe — effect cleans up when component is destroyed
    });
  }

  increment() { this.count.update(n => n + 1); }
  decrement() { this.count.update(n => n - 1); }
  reset()     { this.count.set(0); }
}
```

---

## Example 2: Signals in a Shopping Cart Service

```typescript
// cart.service.ts
import { Injectable, signal, computed } from '@angular/core';

export interface CartItem {
  id: number;
  name: string;
  price: number;
  quantity: number;
}

@Injectable({ providedIn: 'root' })
export class CartService {
  // Private writable signal — items in the cart
  private _items = signal<CartItem[]>([]);

  // Public read-only view
  readonly items = this._items.asReadonly();

  // Computed signals — automatically update when _items changes
  readonly totalItems = computed(() =>
    this._items().reduce((sum, item) => sum + item.quantity, 0)
  );

  readonly totalPrice = computed(() =>
    this._items().reduce((sum, item) => sum + item.price * item.quantity, 0)
  );

  readonly isEmpty = computed(() => this._items().length === 0);

  addItem(item: CartItem) {
    this._items.update(current => {
      const existing = current.find(i => i.id === item.id);
      if (existing) {
        return current.map(i =>
          i.id === item.id ? { ...i, quantity: i.quantity + 1 } : i
        );
      }
      return [...current, { ...item, quantity: 1 }];
    });
  }

  removeItem(id: number) {
    this._items.update(items => items.filter(i => i.id !== id));
  }

  clear() {
    this._items.set([]);
  }
}
```

```typescript
// cart-icon.component.ts — reads signal, auto-updates when cart changes
@Component({
  selector: 'app-cart-icon',
  standalone: true,
  template: `
    <button class="cart-icon">
      🛒 <span class="badge" *ngIf="!cart.isEmpty()">{{ cart.totalItems() }}</span>
    </button>
  `
})
export class CartIconComponent {
  cart = inject(CartService);
}

// cart-summary.component.ts
@Component({
  selector: 'app-cart-summary',
  standalone: true,
  template: `
    <div *ngIf="!cart.isEmpty(); else emptyCart">
      <div *ngFor="let item of cart.items()">
        {{ item.name }} × {{ item.quantity }} = {{ item.price * item.quantity | currency }}
        <button (click)="cart.removeItem(item.id)">Remove</button>
      </div>
      <strong>Total: {{ cart.totalPrice() | currency }}</strong>
    </div>
    <ng-template #emptyCart><p>Your cart is empty.</p></ng-template>
  `
})
export class CartSummaryComponent {
  cart = inject(CartService);
}
```

---

## Example 3: `toSignal` and `toObservable` — Signal/RxJS Interop

```typescript
// product-search.component.ts
import { Component, signal, computed } from '@angular/core';
import { toSignal, toObservable } from '@angular/core/rxjs-interop';
import { debounceTime, distinctUntilChanged, switchMap } from 'rxjs/operators';
import { HttpClient } from '@angular/common/http';
import { FormsModule } from '@angular/forms';
import { AsyncPipe, NgFor } from '@angular/common';

@Component({
  selector: 'app-product-search',
  standalone: true,
  imports: [FormsModule, AsyncPipe, NgFor],
  template: `
    <input [(ngModel)]="query" placeholder="Search products...">

    <p *ngIf="isLoading()">Searching...</p>

    <ul>
      <li *ngFor="let product of results()">
        <strong>{{ product.name }}</strong> — {{ product.price | currency }}
      </li>
    </ul>

    <p *ngIf="results()?.length === 0 && !isLoading()">No results found.</p>
  `
})
export class ProductSearchComponent {
  private http = inject(HttpClient);

  // Writable signal for the search input
  query = signal('');

  isLoading = signal(false);

  // Convert the query signal → Observable → debounce → HTTP call → back to Signal
  results = toSignal(
    toObservable(this.query).pipe(
      debounceTime(300),
      distinctUntilChanged(),
      switchMap(q => {
        if (!q.trim()) return of([]);
        this.isLoading.set(true);
        return this.http.get<Product[]>(`/api/products?search=${q}`).pipe(
          tap(() => this.isLoading.set(false)),
          catchError(() => { this.isLoading.set(false); return of([]); })
        );
      })
    ),
    { initialValue: [] }
  );
}
```

---

## Example 4: Required Inputs + Router Input Binding + Input Transform

```typescript
// product-detail.component.ts
import { Component, Input, OnInit } from '@angular/core';
import { numberAttribute, booleanAttribute } from '@angular/core';
import { NgIf, CurrencyPipe } from '@angular/common';

@Component({
  selector: 'app-product-detail',
  standalone: true,
  imports: [NgIf, CurrencyPipe],
  template: `
    <div class="product-detail">
      <h1>{{ product.name }}</h1>
      <p>{{ product.description }}</p>
      <p>Price: {{ product.price | currency }}</p>

      <!-- From route: /products/:id?tab=reviews&showRelated=true -->
      <p>Product ID: {{ id }}</p>       <!-- auto-converted to number -->
      <p>Active Tab: {{ tab }}</p>      <!-- string from query param -->

      <section *ngIf="showRelated">    <!-- auto-converted to boolean -->
        <h2>Related Products</h2>
      </section>

      <p *ngIf="isAdmin">             <!-- required input from parent -->
        <button (click)="deleteProduct()">Delete Product</button>
      </p>
    </div>
  `
})
export class ProductDetailComponent {
  // Required input — TS compile error if parent doesn't provide it
  @Input({ required: true }) product!: Product;
  @Input({ required: true }) isAdmin!: boolean;

  // Router input binding — auto-bound from route params/query params
  @Input({ transform: numberAttribute }) id = 0;       // /products/:id → number
  @Input() tab = 'overview';                            // ?tab=... → string
  @Input({ transform: booleanAttribute }) showRelated = false; // ?showRelated=true → boolean

  // Resolver data — auto-bound from resolve: { relatedProducts: ... }
  @Input() relatedProducts: Product[] = [];

  deleteProduct() { /* ... */ }
}
```

```typescript
// routes — enable withComponentInputBinding
import { provideRouter, withComponentInputBinding } from '@angular/router';

export const APP_ROUTES: Routes = [
  {
    path: 'products/:id',
    component: ProductDetailComponent,
    resolve: { relatedProducts: relatedProductsResolver }
  }
];

// main.ts
bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(APP_ROUTES, withComponentInputBinding())
    // Now ProductDetailComponent's @Input() fields are auto-populated!
  ]
});
```

---

## Example 5: `takeUntilDestroyed` and `DestroyRef`

```typescript
// dashboard.component.ts — Angular 16 subscription cleanup
import { Component, OnInit, inject } from '@angular/core';
import { takeUntilDestroyed, toSignal } from '@angular/core/rxjs-interop';
import { DestroyRef } from '@angular/core';
import { interval } from 'rxjs';

@Component({
  selector: 'app-dashboard',
  standalone: true,
  template: `
    <div class="dashboard">
      <h2>Live Stats</h2>
      <p>Visitors today: {{ visitorCount() }}</p>
      <p>Server time: {{ serverTime }}</p>
      <p>Notifications: {{ notificationCount }}</p>
    </div>
  `
})
export class DashboardComponent implements OnInit {
  private analyticsService     = inject(AnalyticsService);
  private notificationService  = inject(NotificationService);
  private destroyRef           = inject(DestroyRef);

  serverTime = '';
  notificationCount = 0;

  // Convert Observable → Signal (auto-cleans up with component)
  visitorCount = toSignal(this.analyticsService.getVisitorCount$(), { initialValue: 0 });

  ngOnInit() {
    // Pattern 1: takeUntilDestroyed with explicit destroyRef
    interval(1000).pipe(
      switchMap(() => this.analyticsService.getServerTime()),
      takeUntilDestroyed(this.destroyRef)   // ← auto-cleanup, no ngOnDestroy!
    ).subscribe(time => this.serverTime = time);

    // Pattern 2: DestroyRef.onDestroy for non-Observable cleanup
    const subscription = this.notificationService.count$.subscribe(n => {
      this.notificationCount = n;
    });
    this.destroyRef.onDestroy(() => subscription.unsubscribe());
  }
}
```

```typescript
// data.service.ts — takeUntilDestroyed in an injection context (field initializer)
@Injectable({ providedIn: 'root' })
export class DataService {
  // When called outside a component, you must pass destroyRef
  createPollingStream(destroyRef: DestroyRef) {
    return interval(5000).pipe(
      switchMap(() => this.http.get('/api/status')),
      takeUntilDestroyed(destroyRef)
    );
  }
}

// Usage in component — injection context allows no-arg takeUntilDestroyed
@Component({ standalone: true })
export class StatusComponent {
  // Called during field initialization — injection context is active
  readonly status$ = inject(DataService)
    .poll()
    .pipe(takeUntilDestroyed());  // no arg needed in injection context!
}
```

---

## Example 6: SSR with Non-Destructive Hydration

```typescript
// main.ts — client-side with hydration
import { bootstrapApplication } from '@angular/platform-browser';
import { provideClientHydration } from '@angular/platform-browser';
import { provideRouter } from '@angular/router';
import { AppComponent } from './app/app.component';
import { APP_ROUTES } from './app/app.routes';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(APP_ROUTES),
    provideClientHydration(),   // ← enables non-destructive hydration
  ]
});
```

```typescript
// app.config.server.ts — server-side config
import { mergeApplicationConfig, ApplicationConfig } from '@angular/core';
import { provideServerRendering } from '@angular/platform-server';
import { appConfig } from './app.config';

const serverConfig: ApplicationConfig = {
  providers: [
    provideServerRendering()
  ]
};

export const config = mergeApplicationConfig(appConfig, serverConfig);
```

```typescript
// product-list.component.ts — SSR-safe component
@Component({
  selector: 'app-product-list',
  standalone: true,
  imports: [NgFor, NgIf, AsyncPipe],
  template: `
    <div class="product-list">
      <!-- Server renders this HTML -->
      <!-- Client hydrates without re-rendering — fast! -->
      <app-product-card
        *ngFor="let product of products$ | async"
        [product]="product"
      />
    </div>
  `
})
export class ProductListComponent {
  products$ = inject(ProductService).getProducts();
  // With hydration, server-rendered product cards appear instantly
  // Angular just attaches event listeners — no DOM rebuild
}
```

---

## Example 7: Self-Closing Tags + Input Transform in Design System

```typescript
// icon.component.ts
@Component({
  selector: 'ui-icon',
  standalone: true,
  template: `<span class="icon icon-{{ name }}" [style.fontSize.px]="size"></span>`
})
export class IconComponent {
  @Input({ required: true }) name!: string;
  @Input({ transform: numberAttribute }) size = 24;  // "32" → 32
}

// badge.component.ts
@Component({
  selector: 'ui-badge',
  standalone: true,
  template: `
    <span class="badge badge--{{ variant }}">{{ count }}</span>
  `
})
export class BadgeComponent {
  @Input({ transform: numberAttribute }) count = 0;
  @Input() variant: 'primary' | 'danger' | 'success' = 'primary';
}

// spinner.component.ts
@Component({
  selector: 'ui-spinner',
  standalone: true,
  template: `<div class="spinner spinner--{{ size }}"></div>`
})
export class SpinnerComponent {
  @Input() size: 'sm' | 'md' | 'lg' = 'md';
}
```

```html
<!-- BEFORE Angular 16 — verbose closing tags required -->
<ui-spinner size="sm"></ui-spinner>
<ui-icon name="home" size="20"></ui-icon>
<ui-badge count="5" variant="danger"></ui-badge>

<!-- AFTER Angular 16 — clean self-closing syntax -->
<ui-spinner size="sm" />
<ui-icon name="home" size="20" />
<ui-badge count="5" variant="danger" />

<!-- Works in ngFor too -->
<ui-icon *ngFor="let icon of icons" [name]="icon.name" [size]="icon.size" />
```

# Section 3 — Use Cases

---

## Use Case 1: E-Commerce Platform — Reactive Cart with Signals

**Problem:** A large e-commerce platform had a global shopping cart shared across the header (item count badge), sidebar (cart summary), and checkout page. With RxJS-based state management (BehaviorSubject), developers struggled with:
- Forgetting `async` pipe → stale data
- Multiple `subscribe()` calls with manual `unsubscribe()` for cleanup
- Change detection firing on the whole component tree when cart changed

**Solution with Angular 16 Signals:**

```typescript
// cart.service.ts — signal-based state
@Injectable({ providedIn: 'root' })
export class CartService {
  private _items = signal<CartItem[]>([]);

  readonly items       = this._items.asReadonly();
  readonly totalItems  = computed(() => this._items().reduce((s, i) => s + i.quantity, 0));
  readonly totalPrice  = computed(() => this._items().reduce((s, i) => s + i.price * i.quantity, 0));
  readonly isEmpty     = computed(() => this._items().length === 0);

  add(item: CartItem)    { this._items.update(items => [...items, item]); }
  remove(id: number)     { this._items.update(items => items.filter(i => i.id !== id)); }
  clear()                { this._items.set([]); }
}

// header.component.ts — reads a single computed, renders only when it changes
@Component({ template: `<span class="badge">{{ cart.totalItems() }}</span>` })
export class HeaderComponent { cart = inject(CartService); }

// checkout.component.ts — reads all signals, no subscribe/async pipe
@Component({
  template: `
    <div *ngFor="let item of cart.items()">{{ item.name }}</div>
    <p>Total: {{ cart.totalPrice() | currency }}</p>
    <button [disabled]="cart.isEmpty()" (click)="placeOrder()">Place Order</button>
  `
})
export class CheckoutComponent { cart = inject(CartService); }
```

**Impact:**
- No `subscribe()` calls in components — no memory leaks
- No `async` pipe needed — no template complexity
- Fine-grained updates: only components reading a changed signal re-render
- Code reduced by ~40% vs. equivalent BehaviorSubject pattern

---

## Use Case 2: Content Management System — Router Input Binding for Article Pages

**Problem:** A CMS had article pages with complex routes: `/articles/:slug?tab=comments&page=2`. Every component injected `ActivatedRoute` and manually parsed params in `ngOnInit`, leading to:
- 8 lines of boilerplate per component for param extraction
- Param types were always `string` — required manual `parseInt()` conversion
- Adding a new query param meant changing the template AND the TypeScript AND the route

**Solution with Angular 16 Router Input Binding + Input Transforms:**

```typescript
// article.component.ts — BEFORE (Angular 15)
export class ArticleComponent implements OnInit {
  article!: Article;
  tab = 'content';
  page = 1;

  constructor(private route: ActivatedRoute) {}

  ngOnInit() {
    this.article = this.route.snapshot.data['article'];
    this.tab     = this.route.snapshot.queryParamMap.get('tab') ?? 'content';
    this.page    = parseInt(this.route.snapshot.queryParamMap.get('page') ?? '1', 10);
    // Subscribe to param changes
    this.route.queryParamMap.pipe(takeUntil(this.destroy$)).subscribe(params => {
      this.tab  = params.get('tab') ?? 'content';
      this.page = parseInt(params.get('page') ?? '1', 10);
    });
  }
}

// article.component.ts — AFTER (Angular 16)
export class ArticleComponent {
  @Input({ required: true }) article!: Article;         // from resolver
  @Input() tab = 'content';                             // from ?tab=...
  @Input({ transform: numberAttribute }) page = 1;      // from ?page=2 → number

  // Zero ActivatedRoute injection, zero ngOnInit, zero subscribe
}
```

```typescript
// route config
{
  path: 'articles/:slug',
  component: ArticleComponent,
  resolve: { article: articleResolver }
}

// main.ts
provideRouter(routes, withComponentInputBinding())
```

**Impact:** Removed `ActivatedRoute` from 14 route components. Each component dropped ~12 lines of boilerplate. New route params require adding one `@Input()` line only.

---

## Use Case 3: Analytics Dashboard — Subscription Cleanup with `takeUntilDestroyed`

**Problem:** An analytics dashboard had 6 real-time data streams (visitor count, sales, server metrics, error rate, active sessions, geographic data). Every component had:
- `private destroy$ = new Subject<void>();`
- `ngOnDestroy() { this.destroy$.next(); this.destroy$.complete(); }`
- Memory leak risk if `ngOnDestroy` was accidentally removed

After an audit, 3 components were found to have memory leaks because `takeUntil(this.destroy$)` was missing from one of their streams.

**Solution with Angular 16 `takeUntilDestroyed`:**

```typescript
// analytics-dashboard.component.ts
@Component({ standalone: true, template: `...` })
export class AnalyticsDashboardComponent {
  private destroyRef = inject(DestroyRef);
  private analytics  = inject(AnalyticsService);

  // Convert streams to signals — auto-cleanup built in
  visitorCount  = toSignal(this.analytics.visitors$,    { initialValue: 0 });
  salesTotal    = toSignal(this.analytics.sales$,       { initialValue: 0 });
  errorRate     = toSignal(this.analytics.errors$,      { initialValue: 0 });
  activeSessions= toSignal(this.analytics.sessions$,    { initialValue: 0 });
  geoData       = toSignal(this.analytics.geography$,   { initialValue: [] });

  // For side-effects that need more control
  constructor() {
    this.analytics.alerts$.pipe(
      filter(alert => alert.severity === 'critical'),
      takeUntilDestroyed()   // no arg — injection context active in constructor
    ).subscribe(alert => this.showCriticalAlert(alert));
  }

  private showCriticalAlert(alert: Alert) {
    // Show notification
  }

  // NO ngOnDestroy needed — all cleanup is automatic
}
```

**Impact:** Removed `ngOnDestroy` from 8 components. Eliminated 3 memory leak bugs. Code audit risk reduced to near zero — no manual Subject management.

---

## Use Case 4: News Portal — SSR Hydration for Core Web Vitals

**Problem:** A high-traffic news portal used Angular Universal for SEO. The server rendered the HTML, but when Angular bootstrapped on the client, it **destroyed and rebuilt the DOM** — causing:
- Visible page flicker (content disappears then reappears)
- CLS (Cumulative Layout Shift) score of 0.28 (poor)
- LCP of 3.1 seconds
- Google Search Console flagging the site for poor Core Web Vitals

**Solution with Angular 16 Non-Destructive Hydration:**

```typescript
// main.ts — add provideClientHydration()
bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(routes),
    provideHttpClient(withFetch()),
    provideClientHydration(),    // ← one line change!
  ]
});
```

```typescript
// article-hero.component.ts — unchanged, works with hydration
@Component({
  standalone: true,
  imports: [NgOptimizedImage],
  template: `
    <!-- Server renders this HTML. Client attaches listeners — no DOM rebuild. -->
    <img [ngSrc]="article.heroImage" width="1200" height="630" priority [alt]="article.title">
    <h1>{{ article.title }}</h1>
    <time>{{ article.publishedAt | date }}</time>
    <button (click)="share()">Share</button>   <!-- event listener attached on hydration -->
  `
})
export class ArticleHeroComponent {
  @Input({ required: true }) article!: Article;
  share() { navigator.share({ title: this.article.title, url: location.href }); }
}
```

**Results:**
| Metric | Before (v15 SSR) | After (v16 Hydration) |
|---|---|---|
| LCP | 3.1s | 1.4s |
| CLS | 0.28 | 0.03 |
| Page flicker | Visible | Gone |
| JS parse + execute | 100% (baseline) | ~55% (less DOM work) |
| Google Search ranking | Declining | Improved within 30 days |

---

## Use Case 5: Financial App — Design System with Self-Closing Tags + Required Inputs

**Problem:** A financial services app had a complex component library with 45+ UI components. Template code was verbose and error-prone:
- Missing closing tags caused subtle layout bugs
- Components with required props had no compile-time safety — runtime null errors in production
- `@Input() amount: number` could be accidentally passed as a string `"1500"` from route params

**Solution with Angular 16 Self-Closing Tags + Required Inputs + Transform:**

```typescript
// amount-display.component.ts
@Component({
  selector: 'fin-amount',
  standalone: true,
  template: `
    <span [class]="'amount amount--' + (value >= 0 ? 'positive' : 'negative')">
      {{ value | currency:currency:'symbol':'1.2-2' }}
    </span>
  `
})
export class AmountDisplayComponent {
  @Input({ required: true, transform: numberAttribute }) value!: number;
  @Input({ required: true }) currency!: string;
}

// account-card.component.ts
@Component({
  selector: 'fin-account-card',
  standalone: true,
  template: `
    <div class="account-card">
      <h3>{{ account.name }}</h3>
      <fin-amount [value]="account.balance" [currency]="account.currency" />
      <fin-badge [count]="account.pendingTransactions" variant="warning" />
      <fin-icon name="chevron-right" size="16" />
    </div>
  `
})
export class AccountCardComponent {
  @Input({ required: true }) account!: Account;
  // Compile error if parent doesn't pass [account] — caught before production!
}
```

```html
<!-- Dashboard template — clean, self-closing component syntax -->
<fin-account-card *ngFor="let acc of accounts" [account]="acc" />
<fin-spinner *ngIf="isLoading" size="lg" />
<fin-empty-state *ngIf="!isLoading && !accounts.length" message="No accounts found" />
```

**Impact:**
- 3 production null-reference bugs eliminated by `required: true` at compile time
- `numberAttribute` transform eliminated 6 `parseInt()` calls and associated `NaN` edge cases
- Template code 30% shorter with self-closing syntax

---

## Use Cases Summary Table

| Use Case | Feature Used | Business Value |
|---|---|---|
| E-Commerce Cart | Signals + Computed | 40% less code, no memory leaks, fine-grained re-render |
| CMS Article Pages | Router Input Binding + Transform | Removed `ActivatedRoute` from 14 components, ~12 lines saved each |
| Analytics Dashboard | `takeUntilDestroyed` + `toSignal` | 3 memory leak bugs fixed, `ngOnDestroy` removed from 8 components |
| News Portal | SSR Hydration | LCP 3.1s→1.4s, CLS 0.28→0.03, no page flicker |
| Financial Design System | Required Inputs + Self-closing tags | 3 prod bugs eliminated, 30% shorter templates |

# Section 4 — Interview Q&A

---

## Basic Level Questions

---

### Q1. What is the biggest new feature in Angular 16?

**Answer:**
**Angular Signals** — a new reactive primitive that provides fine-grained reactivity. Signals allow state to be tracked and automatically push updates to consumers (templates, computed values, effects) without Zone.js change detection cycles.

Three core primitives:
```typescript
const count   = signal(0);                          // writable state
const doubled = computed(() => count() * 2);        // derived, auto-updates
effect(() => console.log('Count:', count()));        // side-effect on change
```

Signals are in **Developer Preview** in Angular 16 and become stable in Angular 17.

---

### Q2. What is a `computed` signal? How does it differ from a regular signal?

**Answer:**
- A **`signal`** holds a value you can **read and write** (`set`, `update`, `mutate`)
- A **`computed`** is **read-only** and **derived** from one or more other signals — it recalculates automatically when its dependencies change

```typescript
const price    = signal(100);
const quantity = signal(3);

// computed — read-only, auto-recalculates when price or quantity changes
const total = computed(() => price() * quantity());

console.log(total());  // 300
price.set(150);
console.log(total());  // 450 — auto-updated!

// Cannot do: total.set(...)  → compile error
```

Computed signals are **lazy**: they only recalculate when they are read AND a dependency changed.

---

### Q3. What is a Required Input in Angular 16?

**Answer:**
`@Input({ required: true })` marks a component input as mandatory. If a parent template omits it, TypeScript's template type checker reports a **compile-time error** — caught during `ng build` or `ng serve`, not at runtime.

```typescript
// In the child component:
@Input({ required: true }) userId!: string;  // required
@Input() theme = 'light';                    // optional with default

// In the parent template:
<app-user-profile />                    // ❌ Error: userId is required
<app-user-profile [userId]="'u-123'" /> // ✅ OK
```

---

### Q4. What is `withComponentInputBinding()` and what does it do?

**Answer:**
`withComponentInputBinding()` is a router feature that automatically maps route data (path params, query params, resolver data, static data) to matching `@Input()` properties on a routed component:

```typescript
// Activate in main.ts
provideRouter(routes, withComponentInputBinding())

// Route: /products/:id?tab=details
// Component:
@Input() id!: string;          // bound from :id path param
@Input() tab = 'overview';     // bound from ?tab= query param
@Input() product!: Product;    // bound from resolve: { product: resolver }
```

Eliminates the need to inject and manually parse `ActivatedRoute`.

---

### Q5. What is `takeUntilDestroyed`? What problem does it solve?

**Answer:**
`takeUntilDestroyed()` is an RxJS operator that **automatically completes an Observable when the host component/directive/pipe is destroyed**. It eliminates the classic `Subject + takeUntil + ngOnDestroy` cleanup pattern:

```typescript
// BEFORE — boilerplate every time
private destroy$ = new Subject<void>();
ngOnInit()    { this.data$.pipe(takeUntil(this.destroy$)).subscribe(...); }
ngOnDestroy() { this.destroy$.next(); this.destroy$.complete(); }

// AFTER — Angular 16
ngOnInit() {
  this.data$.pipe(takeUntilDestroyed(this.destroyRef)).subscribe(...);
}
// No ngOnDestroy!
```

---

## Intermediate Level Questions

---

### Q6. What is the difference between `signal.set()`, `signal.update()`, and `signal.mutate()`?

**Answer:**

| Method | Use when | Example |
|---|---|---|
| `.set(value)` | Replacing with a completely new value | `count.set(0)` |
| `.update(fn)` | Computing new value from current value | `count.update(n => n + 1)` |
| `.mutate(fn)` | Modifying an object/array in place | `items.mutate(arr => arr.push(item))` |

```typescript
const count = signal(5);
count.set(10);                  // 10 — direct replacement
count.update(n => n * 2);      // 20 — based on current value

const cart = signal<CartItem[]>([]);
cart.mutate(items => items.push({ id: 1, name: 'Shoe' }));  // in-place mutation

// NOTE: mutate() is deprecated in later versions — prefer update() with immutability
cart.update(items => [...items, { id: 1, name: 'Shoe' }]);
```

---

### Q7. How do you convert between Signals and Observables in Angular 16?

**Answer:**
Angular 16 provides two interop utilities in `@angular/core/rxjs-interop`:

```typescript
import { toSignal, toObservable } from '@angular/core/rxjs-interop';

// 1. Observable → Signal (most common: HTTP calls, WebSocket streams)
const products = toSignal(this.http.get<Product[]>('/api/products'), {
  initialValue: []      // required if Observable is async
});
// Use in template: {{ products() }} — no async pipe needed

// 2. Signal → Observable (for combining with RxJS operators)
const query = signal('');
const results$ = toObservable(query).pipe(
  debounceTime(300),
  switchMap(q => this.http.get('/api/search?q=' + q))
);
const results = toSignal(results$, { initialValue: [] });
```

`toSignal` must be called inside an **injection context** (constructor, field initializer, or `runInInjectionContext`).

---

### Q8. What are `numberAttribute` and `booleanAttribute` transforms in Angular 16?

**Answer:**
These are built-in transform functions for `@Input({ transform: ... })` that coerce HTML attribute strings to their correct types:

```typescript
import { numberAttribute, booleanAttribute } from '@angular/core';

@Component({ selector: 'app-progress' })
export class ProgressComponent {
  // HTML attributes are always strings — these transforms handle conversion
  @Input({ transform: numberAttribute })  value = 0;    // "75" → 75
  @Input({ transform: numberAttribute })  max   = 100;  // "100" → 100
  @Input({ transform: booleanAttribute }) striped = false; // "true"/""/"striped" → true
}
```

```html
<!-- Works correctly with these transforms -->
<app-progress value="75" max="100" striped></app-progress>
```

Without transforms, `value` would be the string `"75"` inside the component.

---

### Q9. How does `effect()` differ from `computed()`?

**Answer:**

| | `computed()` | `effect()` |
|---|---|---|
| Returns | A value (read-only signal) | void |
| Purpose | Derive data from other signals | Perform side effects |
| Lazy | Yes — only runs when read | No — runs immediately on dependency change |
| Use for | Calculations, formatting | Logging, DOM updates, localStorage, analytics |

```typescript
const count = signal(0);

// computed — produces a value
const label = computed(() => count() === 1 ? '1 item' : `${count()} items`);

// effect — side effect, no return value
effect(() => {
  localStorage.setItem('count', count().toString());
  // Runs once immediately, then whenever count() changes
});
```

**Important:** Never write signals inside `effect()` by default — use `allowSignalWrites: true` only when necessary, or restructure to use `computed`.

---

### Q10. What is non-destructive hydration and how is it different from previous Angular SSR?

**Answer:**

**Previous SSR (Angular ≤ 15):**
1. Server renders HTML → browser receives it
2. Browser displays server HTML
3. Angular bootstraps → **destroys server DOM**
4. Angular **rebuilds** the entire DOM → visible flicker

**Angular 16 Non-Destructive Hydration:**
1. Server renders HTML → browser receives it
2. Browser displays server HTML (immediately visible)
3. Angular bootstraps → **traverses existing DOM** (no rebuild)
4. Angular **attaches event listeners** to existing DOM nodes
5. No flicker — page is interactive without visual change

```typescript
// Enable with one line:
provideClientHydration()
```

**Result:** LCP improves because the browser doesn't have to wait for JavaScript to render content — it was already there from the server.

---

## Advanced Level Questions

---

### Q11. How does Angular's Signal reactivity model compare to Zone.js change detection?

**Answer:**

**Zone.js model (Angular 1–15):**
- Zone.js patches browser APIs (setTimeout, Promise, XHR, events)
- Any async event → Zone.js triggers → Angular checks the **entire component tree** (`ApplicationRef.tick()`)
- Even components with no data change get checked
- **Push-based**: framework pushes checks to all components

**Signals model (Angular 16+):**
- Signals track which **template expressions** read them
- When a signal changes → only **components whose template reads that signal** are marked dirty
- Angular re-renders only those components
- **Pull-based**: component re-renders only when needed

```typescript
// Signal change only re-renders components that read count()
const count = signal(0);

// Component A reads count() → re-renders when count changes
// Component B does NOT read count() → never re-renders on count change
```

**Future vision:** Signals will eventually allow Angular to run **without Zone.js** entirely, reducing bundle size by ~12KB and eliminating the overhead of monkey-patching.

---

### Q12. What are the rules for calling `toSignal()` and why must it be in an injection context?

**Answer:**
`toSignal()` internally uses `inject(DestroyRef)` to know when to unsubscribe from the Observable. `inject()` can only be called in an **injection context**.

**Valid injection contexts:**
```typescript
@Component({ ... })
export class MyComponent {
  // 1. Field initializer — ✅
  data = toSignal(this.service.data$, { initialValue: [] });

  constructor() {
    // 2. Constructor body — ✅
    this.other = toSignal(this.service.other$);
  }
}
```

**Outside injection context:**
```typescript
// 3. ngOnInit — ❌ ERROR: injection context not active
ngOnInit() {
  this.data = toSignal(this.service.data$);  // Error!
}

// Fix: use runInInjectionContext
ngOnInit() {
  this.data = runInInjectionContext(this.injector, () =>
    toSignal(this.service.data$)
  );
}
```

**Options for `toSignal`:**
- `initialValue` — value to return before the Observable emits (prevents undefined)
- `requireSync` — throws if Observable doesn't emit synchronously (no initialValue needed)
- `injector` — custom injector for use outside injection context

---

### Q13. How do Signals handle object/array mutations? What is the recommended pattern?

**Answer:**
Signals use **reference equality** to detect changes. Mutating an object in place (without creating a new reference) will NOT trigger dependents:

```typescript
const user = signal({ name: 'Alice', age: 30 });

// ❌ WRONG — mutation, reference unchanged → no update
user().age = 31;

// ✅ CORRECT — new object reference → all dependents update
user.update(u => ({ ...u, age: 31 }));

// For arrays:
const items = signal<string[]>([]);

// ❌ WRONG
items().push('newItem');

// ✅ CORRECT
items.update(arr => [...arr, 'newItem']);

// signal.mutate() — the exception (mutates in place and notifies):
// NOTE: mutate() was available in v16 but deprecated later — avoid it
items.mutate(arr => arr.push('newItem'));  // works in v16 but not idiomatic
```

**Best practice:** Always use immutable updates (`spread`, `filter`, `map`) with `.update()`.

---

### Q14. How do you use signals with the `OnPush` change detection strategy?

**Answer:**
Signals work **perfectly** with `ChangeDetectionStrategy.OnPush`. In fact, the combination delivers the best performance:

```typescript
@Component({
  changeDetection: ChangeDetectionStrategy.OnPush,
  template: `
    <h2>{{ user().name }}</h2>          <!-- reads a signal -->
    <p>{{ formatDate(user().joined) }}</p>
  `
})
export class UserCardComponent {
  private userService = inject(UserService);

  // Signal in template — Angular marks component dirty when user signal changes
  user = toSignal(this.userService.currentUser$, { requireSync: true });
}
```

**Why this combination is best:**
- `OnPush` prevents default Zone.js-triggered checks
- Signal reads in the template create a precise dependency tracking
- Only when `user` signal changes → component is marked for re-render
- **Result:** Component re-renders ONLY when its specific signal dependency changes

---

### Q15. What happens when `provideClientHydration()` encounters a DOM mismatch between server and client?

**Answer:**
If the server-rendered HTML doesn't match what the client would render (e.g., a component renders differently based on `isPlatformBrowser`), Angular's hydration will detect the mismatch and log a warning, then fall back to **client-side re-rendering** for that subtree.

**Common causes of mismatch:**
1. Using `Date.now()`, `Math.random()`, or UUIDs without seeding them consistently
2. Rendering different content for browser vs. server
3. Directly accessing browser-only APIs (`window`, `document`) without `isPlatformBrowser` guard

**How to avoid mismatches:**
```typescript
// ❌ Causes mismatch — server/client will have different random IDs
@Component({ template: `<div [id]="randomId">...</div>` })
export class MyComponent {
  randomId = Math.random().toString(36);  // different on server vs client!
}

// ✅ Use a deterministic ID or skip ID rendering on server
@Component({ template: `<div [id]="id">...</div>` })
export class MyComponent {
  id = inject(ElementRef).nativeElement.getAttribute('id') ?? 'component-1';
}
```

In development mode, Angular 16 logs detailed warnings for each mismatch to help debugging.

---

## Scenario-Based Questions

---

### Q16. Your team has an existing Angular 15 app with 50+ components using `BehaviorSubject` for state management. How would you incrementally migrate to Signals without breaking the app?

**Answer:**
**Incremental migration strategy — keep RxJS and Signals coexisting:**

```typescript
// Step 1: Convert leaf services first (no dependencies on other services)
// BEFORE
export class ThemeService {
  private _theme = new BehaviorSubject<'light' | 'dark'>('light');
  theme$ = this._theme.asObservable();
  setTheme(t: 'light' | 'dark') { this._theme.next(t); }
}

// AFTER — Signals, backward-compatible with RxJS consumers via toObservable
export class ThemeService {
  private _theme = signal<'light' | 'dark'>('light');
  readonly theme  = this._theme.asReadonly();
  readonly theme$ = toObservable(this._theme);  // ← existing consumers still work
  setTheme(t: 'light' | 'dark') { this._theme.set(t); }
}

// Step 2: Update templates to use signal directly (no async pipe)
// BEFORE: <body [class]="themeService.theme$ | async">
// AFTER:  <body [class]="themeService.theme()">

// Step 3: Convert components one by one — mixed approach works:
// Some components use toSignal() on existing Observables
// Others read signals directly
// RxJS-based and Signal-based code coexist via toSignal/toObservable

// Step 4: After all consumers are migrated, remove toObservable wrappers
```

---

### Q17. A component has 10 `@Input()` properties from a route. Some come from params, some from query params, some from resolver. How do you set this up in Angular 16?

**Answer:**
```typescript
// product-detail.component.ts
@Component({ standalone: true })
export class ProductDetailComponent {
  // From route: /category/:catId/products/:productId
  @Input({ required: true }) catId!: string;
  @Input({ required: true }) productId!: string;

  // From resolve: { product: productResolver, related: relatedResolver }
  @Input({ required: true }) product!: Product;
  @Input() related: Product[] = [];

  // From query params: ?tab=specs&page=2&showReviews=true
  @Input() tab = 'overview';
  @Input({ transform: numberAttribute })  page = 1;
  @Input({ transform: booleanAttribute }) showReviews = false;

  // From static route data: data: { breadcrumb: 'Product Details' }
  @Input() breadcrumb = '';
}

// Route config
{
  path: 'category/:catId/products/:productId',
  component: ProductDetailComponent,
  resolve: {
    product: productResolver,
    related: relatedProductsResolver
  },
  data: { breadcrumb: 'Product Details' }
}

// main.ts
provideRouter(routes, withComponentInputBinding())
```

All 10 inputs are automatically populated — zero `ActivatedRoute` injection.

---

### Q18. How do you write a unit test for a component that uses Signals?

**Answer:**
Signal-based components test almost identically to regular components — Signals are synchronous, so no `async`/`fakeAsync` is needed for signal updates:

```typescript
// counter.component.spec.ts
describe('CounterComponent', () => {
  let component: CounterComponent;
  let fixture: ComponentFixture<CounterComponent>;

  beforeEach(async () => {
    await TestBed.configureTestingModule({
      imports: [CounterComponent]   // standalone component
    }).compileComponents();

    fixture = TestBed.createComponent(CounterComponent);
    component = fixture.componentInstance;
    fixture.detectChanges();
  });

  it('should start at 0', () => {
    expect(component.count()).toBe(0);
  });

  it('should increment when increment() is called', () => {
    component.increment();
    expect(component.count()).toBe(1);
    // Computed signals update synchronously
    expect(component.doubleCount()).toBe(2);
  });

  it('should update template when signal changes', () => {
    component.increment();
    fixture.detectChanges();   // trigger change detection
    const h2 = fixture.nativeElement.querySelector('h2');
    expect(h2.textContent).toContain('1');
  });

  it('should reset to 0', () => {
    component.increment();
    component.increment();
    component.reset();
    expect(component.count()).toBe(0);
  });
});
```

---

### Q19. A developer accidentally wrote `signal.update(s => { s.items.push(item); return s; })`. What is the problem and how do you fix it?

**Answer:**
**Problem:** `signal.update()` receives the **same object reference** and returns it. Since the reference hasn't changed, Angular's signal system doesn't know the value changed — dependents are **not notified**. The UI won't update.

```typescript
// ❌ WRONG — same reference returned
const state = signal({ items: [], count: 0 });

state.update(s => {
  s.items.push(newItem);  // mutates in place
  s.count++;
  return s;               // same reference → NO UPDATE NOTIFICATION
});
```

```typescript
// ✅ CORRECT — new object reference
state.update(s => ({
  ...s,
  items: [...s.items, newItem],   // new array
  count: s.count + 1
}));

// ✅ Also correct — use set() with spread
state.set({
  ...state(),
  items: [...state().items, newItem],
  count: state().count + 1
});
```

---

### Q20. When would you choose Signals over RxJS? When would you keep RxJS?

**Answer:**

**Choose Signals when:**
- Managing simple component/service state (loading, current user, cart, UI flags)
- Deriving values from other state (`computed`)
- Binding state to templates directly
- You want synchronous, always-available state

```typescript
// Perfect for Signals
const isLoading = signal(false);
const user      = signal<User | null>(null);
const cartCount = computed(() => cart().length);
```

**Keep RxJS when:**
- Handling async event streams (WebSocket, SSE, polling)
- HTTP requests with complex operators (retry, debounce, switchMap, mergeMap)
- Complex multi-step async workflows
- Combining multiple async sources (combineLatest, zip, forkJoin)

```typescript
// Better with RxJS
const search$ = searchInput$.pipe(
  debounceTime(300),
  distinctUntilChanged(),
  switchMap(q => this.http.get(`/search?q=${q}`)),
  retry(3),
  catchError(e => EMPTY)
);

// Bridge with toSignal when you need it in the template
const searchResults = toSignal(search$, { initialValue: [] });
```

**Best practice (Angular 16+):** Use Signals for state, RxJS for streams, bridge with `toSignal`/`toObservable`.

---

## Quick Reference Card

### Angular 16 Key Facts for Interviews

| Question | Answer |
|---|---|
| Release date | May 3, 2023 |
| Signals status | Developer Preview (stable in v17) |
| Required input syntax | `@Input({ required: true }) name!: Type` |
| Router input binding | `provideRouter(routes, withComponentInputBinding())` |
| SSR hydration | `provideClientHydration()` |
| Subscription cleanup | `takeUntilDestroyed(destroyRef)` |
| Build system preview | Vite + esbuild (`application` builder) |
| Input type coercion | `numberAttribute`, `booleanAttribute` |
| Self-closing tags | `<my-comp />` valid in templates |
| TypeScript versions | 4.9.x – 5.0.x |
| Signal interop | `toSignal(obs$)`, `toObservable(signal)` |

### Before vs After Cheatsheet

```
BehaviorSubject state         →  signal() + computed()
async pipe in template        →  signal reads — {{ data() }}
takeUntil(destroy$)          →  takeUntilDestroyed(destroyRef)
ngOnDestroy + Subject         →  DestroyRef.onDestroy()
ActivatedRoute.snapshot.data  →  @Input() name!: Type (+ withComponentInputBinding)
parseInt(route.param)         →  @Input({ transform: numberAttribute }) param!: number
<my-comp></my-comp>           →  <my-comp /> (self-closing)
@Input() value?: string       →  @Input({ required: true }) value!: string
ng serve (Webpack)            →  ng serve (Vite + esbuild — ~67% faster)
Full DOM rebuild on hydration →  provideClientHydration() — reuse server DOM
```